# ST graph transfer-learning pipeline

Run the sections from top to bottom.

1. Initialize the project
2. Build spatial graphs from coordinates and cell types
3. Optionally train an ST baseline from scratch
4. Fine-tune a frozen CODEX model
5. Evaluate the fine-tuned model


## 1. Initialize the project

Input tables belong under `data/st_gnn/`, and the CODEX checkpoint belongs at `checkpoints/codex_pretrained.pt`. All locations are relative to the repository root.


In [ ]:
from pathlib import Path
import runpy
import sys

candidates = [Path.cwd() / 'ST_GNN', Path.cwd()]
PIPELINE_DIR = next(path for path in candidates if (path / 'config.py').exists() and (path / '03_finetune_frozen.py').exists())
sys.path.insert(0, str(PIPELINE_DIR.resolve()))
print(f'Pipeline directory: {PIPELINE_DIR}')


## 2. Build spatial graphs

Each region requires `{REGION_ID}.cell_data.csv` and `{REGION_ID}.cell_types.csv`. The label table must contain `REGION_ID` and `MET_type`.


In [ ]:
runpy.run_path(str(PIPELINE_DIR / '01_build_graphs.py'), run_name='__main__')


## 3. Train an optional from-scratch baseline

This stage is not required for transfer learning. It trains a new GIN directly on the ST graphs using a stratified 70/30 region split.


In [ ]:
runpy.run_path(str(PIPELINE_DIR / '02_train_baseline.py'), run_name='__main__')


## 4. Fine-tune the frozen CODEX model

The CODEX checkpoint supplies the pretrained GIN message-passing weights. Cell-type embeddings, feature projections, and both prediction heads are reinitialized for the ST dataset. Message-passing parameters remain frozen during fine-tuning.


In [ ]:
runpy.run_path(str(PIPELINE_DIR / '03_finetune_frozen.py'), run_name='__main__')


## 5. Evaluate on held-out regions

Predictions are averaged across all center cells in each held-out region. The script writes `outputs/st_gnn/analysis/confusion_matrix.csv`.


In [ ]:
runpy.run_path(str(PIPELINE_DIR / '04_evaluate.py'), run_name='__main__')
